# BCEDF: Breast Cancer Early Detection Framework
## Full Training Pipeline for M.S. Thesis

This notebook implements the complete BCEDF framework as described in the thesis:
- 3 Datasets: MIAS, BreakHis, INbreast
- 3 Architectures: Custom CNN, ResNet50, DenseNet121
- Two-phase fine-tuning (thesis Chapter 6)
- Grad-CAM explainability (thesis Chapter 7)
- Comprehensive evaluation metrics

**Runtime setup**: Go to Runtime → Change runtime type → T4 GPU

In [ ]:
# Mount Google Drive for dataset access
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ========== CONFIGURATION - EDIT THESE PATHS ==========
PROJECT_DIR = '/content/drive/MyDrive/breast_cancer_project'

# Dataset paths on your Google Drive
DATASETS = {
    'breakhis': '/content/drive/MyDrive/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast',
    'mias': '/content/drive/MyDrive/mias',
    'inbreast': '/content/drive/MyDrive/inbreast'
}

# Which architectures to run (comment out to skip)
ARCHITECTURES = ['cnn', 'resnet50', 'densenet121']

# Which dataset configs to run
DATASET_CONFIGS = [
    ['breakhis'],
    ['mias'],
    ['inbreast'],
    ['breakhis', 'mias', 'inbreast'],  # combined
]

EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001
# ======================================================

In [ ]:
# Copy project files to Colab
import shutil, os
src = PROJECT_DIR
dst = '/content/breast_cancer_project'
print(f'Source: {src}')
print(f'Source exists: {os.path.exists(src)}')
if not os.path.exists(src):
    src = '/content/drive/MyDrive/breast_cancer_project'
    print(f'Fallback: {src}, exists: {os.path.exists(src)}')
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst)
print('Project files copied')

In [ ]:
# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy pandas scikit-learn scipy matplotlib seaborn opencv-python albumentations tqdm tensorboard PyYAML easydict pydicom
print('Dependencies installed')

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'Device count: {torch.cuda.device_count()}')

In [ ]:
import sys
sys.path.insert(0, '/content/breast_cancer_project')

import torch.nn as nn
import argparse
import torch
import numpy as np
import os
import json
import yaml
from datetime import datetime
from easydict import EasyDict as edict

from src.models.model_factory import create_model, ModelFactory, CustomCNN
from src.datasets.dataloader import DataLoaderFactory
from src.training.trainer import Trainer
from src.config import Config
from src.utils.logger import Logger
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.visualizer import Visualizer
from src.evaluation.gradcam import GradCAMView

assert issubclass(CustomCNN, nn.Module), 'CustomCNN not found'
print('CustomCNN verified')

parser = argparse.ArgumentParser()
parser.add_argument('--model', choices=['cnn', 'efficientnet_b3', 'efficientnet_b4', 'densenet121', 'resnet50', 'mobilenet_v3_small'])
assert 'cnn' in parser.parse_args(['--model', 'cnn']).model
print('CNN option verified')

print('All imports successful')

In [ ]:
def create_config_for_run(model_name, dataset_names, epochs=50, batch_size=32, lr=0.001):
    """Create a config dict for a specific run"""
    cfg = {
        'data': {
            'breakhis': {
                'path': DATASETS['breakhis'],
                'use': 'breakhis' in dataset_names,
                'magnification': [40, 100, 200, 400]
            },
            'inbreast': {
                'path': DATASETS['inbreast'],
                'use': 'inbreast' in dataset_names
            },
            'mias': {
                'path': DATASETS['mias'],
                'use': 'mias' in dataset_names
            }
        },
        'training': {
            'batch_size': batch_size,
            'epochs': epochs,
            'early_stop_patience': 10,
            'learning_rate': lr,
            'min_lr': 1e-6,
            'weight_decay': 0.0001,
            'num_workers': 2,
            'mixed_precision': True,
            'label_smoothing': 0.1,
            'k_folds': 0,
            'val_split': 0.15,
            'test_split': 0.10,
            'seed': 42,
            'gradient_clip': 1.0
        },
        'model': {
            'name': model_name,
            'pretrained': model_name != 'cnn',
            'num_classes': 2,
            'dropout': 0.3,
            'use_ensemble': False,
        },
        'augmentation': {
            'img_size': 224,
            'clahe_clip_limit': 2.0,
            'clahe_grid_size': 8,
            'mixup_alpha': 0.2,
            'cutmix_alpha': 1.0,
            'mixup_prob': 0.5,
            'rotation': 30,
            'brightness': 0.2,
            'contrast': 0.2,
            'saturation': 0.2,
            'hue': 0.1
        },
        'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        'output': {
            'checkpoint_dir': f'/content/outputs/{model_name}/checkpoints',
            'log_dir': f'/content/outputs/{model_name}/logs',
            'plot_dir': f'/content/outputs/{model_name}/plots',
            'model_dir': f'/content/outputs/{model_name}/models',
            'gradcam_dir': f'/content/outputs/{model_name}/gradcam'
        }
    }
    cfg_obj = edict(cfg)
    cfg_obj.cfg = cfg_obj
    return cfg_obj


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print('Helper functions defined')

In [ ]:
def train_and_evaluate(model_name, dataset_names, results_dict):
    """Train a model on given datasets and return results"""
    ds_label = '+'.join(dataset_names)
    run_name = f'{model_name}_{ds_label}'
    print(f'\n{"="*60}')
    print(f'RUN: {run_name}')
    print(f'{"="*60}')

    cfg = create_config_for_run(model_name, dataset_names, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE)
    set_seed(cfg.training.seed)

    # Create output dirs
    for d in ['checkpoint_dir', 'log_dir', 'plot_dir', 'model_dir', 'gradcam_dir']:
        os.makedirs(cfg.output[d], exist_ok=True)

    # Create model
    model = create_model(cfg)
    model = model.to(cfg.device)
    print(f'Model: {model_name} | Params: {sum(p.numel() for p in model.parameters()):,}')

    # Load data
    loader_factory = DataLoaderFactory(cfg)
    train_loader, val_loader, test_loader = loader_factory.get_dataloaders(dataset_names)
    print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')

    # Train
    trainer = Trainer(model, cfg)
    best_val_acc = trainer.fit(train_loader, val_loader)
    print(f'Best val acc: {best_val_acc:.4f}')

    # Test evaluation
    test_loss, test_acc, test_info = trainer.validate(test_loader)

    # Full metrics
    metrics = ClassificationMetrics(num_classes=2)
    all_outputs, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(cfg.device)
            labels = labels.to(cfg.device)
            outputs = torch.softmax(model(images), dim=1)
            all_outputs.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_outputs = np.concatenate(all_outputs)
    all_labels = np.concatenate(all_labels)
    all_preds = all_outputs.argmax(axis=1)
    report = metrics.compute(all_labels, all_preds, all_outputs)

    print(f'Test Acc: {report["accuracy"]:.4f} | F1: {report["f1_score"]:.4f} | AUC: {report["auc_roc"]:.4f}')
    print(f'Sens: {report["sensitivity"]:.4f} | Spec: {report["specificity"]:.4f}')

    # Save plots
    viz = Visualizer(cfg.output.plot_dir)
    try:
        viz.plot_confusion_matrix(np.array(report['confusion_matrix']), class_names=['benign', 'malignant'])
        viz.plot_roc_curve(all_labels, all_outputs, class_names=['benign', 'malignant'])
        viz.plot_training_history(trainer.train_losses, trainer.val_losses, trainer.train_accs, trainer.val_accs)
    except Exception as e:
        print(f'Plot warning: {e}')

    # Grad-CAM on test samples
    gradcam_dir = cfg.output.gradcam_dir
    try:
        target_layers = []
        if model_name == 'resnet50':
            target_layers = ['layer4']
        elif model_name == 'densenet121':
            target_layers = ['features']
        elif model_name == 'cnn':
            target_layers = ['features']
        elif model_name == 'mobilenet_v3_small':
            target_layers = ['features']

        gradcam_view = GradCAMView(model, target_layers, cfg.device)
        import cv2
        n_samples = min(10, len(test_loader.dataset))
        for idx in range(n_samples):
            img_tensor = test_loader.dataset[idx][0].unsqueeze(0).to(cfg.device)
            heatmaps = gradcam_view.generate_heatmap(img_tensor)
            img_np = img_tensor.squeeze().cpu().numpy().transpose(1, 2, 0)
            img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
            img_np = (img_np * 255).astype(np.uint8)
            for layer_name, heatmap in heatmaps.items():
                save_path = os.path.join(gradcam_dir, f'sample_{idx}_{layer_name}.png')
                gradcam_view.save_heatmap(img_np, heatmap, save_path)
        print(f'Grad-CAM saved: {n_samples} samples')
    except Exception as e:
        print(f'Grad-CAM warning: {e}')

    # Save model
    model_path = os.path.join(cfg.output.model_dir, f'{model_name}_final.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': cfg,
        'test_accuracy': report['accuracy'],
        'test_report': report
    }, model_path)
    print(f'Model saved to {model_path}')

    # Store results
    results_dict[run_name] = {
        'model': model_name,
        'datasets': ds_label,
        'accuracy': float(report['accuracy']),
        'f1_score': float(report['f1_score']),
        'auc_roc': float(report['auc_roc']),
        'sensitivity': float(report['sensitivity']),
        'specificity': float(report['specificity']),
        'precision': float(report.get('precision', 0)),
        'recall': float(report.get('recall', 0)),
        'confusion_matrix': report['confusion_matrix'].tolist() if hasattr(report['confusion_matrix'], 'tolist') else report['confusion_matrix'],
        'best_val_acc': float(best_val_acc),
    }
    return results_dict


print('Training function defined')

In [ ]:
# ========== RUN THE FULL PIPELINE ==========
results = {}
start_time = datetime.now()
print(f'Start time: {start_time}')
print(f'Architectures: {ARCHITECTURES}')
print(f'Dataset configs: {DATASET_CONFIGS}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}')

for model_name in ARCHITECTURES:
    for ds_names in DATASET_CONFIGS:
        try:
            results = train_and_evaluate(model_name, ds_names, results)
        except Exception as e:
            import traceback
            traceback.print_exc()
            print(f'FAILED: {model_name} on {ds_names}: {e}')

elapsed = datetime.now() - start_time
print(f'\n{"="*60}')
print(f'ALL RUNS COMPLETE | Total time: {elapsed}')
print(f'{"="*60}')

In [ ]:
# Save results
results_path = '/content/outputs/all_results.json'
os.makedirs('/content/outputs', exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

# Also copy to Drive
drive_results_path = os.path.join(PROJECT_DIR, 'outputs', 'all_results.json')
os.makedirs(os.path.dirname(drive_results_path), exist_ok=True)
shutil.copy(results_path, drive_results_path)

# Copy all outputs to Drive
drive_output_dir = os.path.join(PROJECT_DIR, 'outputs', 'colab_results')
if os.path.exists(drive_output_dir):
    shutil.rmtree(drive_output_dir)
shutil.copytree('/content/outputs', drive_output_dir)

print(f'Results saved to Drive: {drive_output_dir}')

In [ ]:
# Display summary table as per thesis Table 6.1
print(f'{"="*100}')
print(f'{"BCEDF Summary Table (cf. Thesis Table 6.1)":^100}')
print(f'{"="*100}')
print(f'{"Model":<15} {"Dataset":<15} {"Acc":<8} {"F1":<8} {"AUC":<8} {"Sens":<8} {"Spec":<8}')
print(f'{"-"*70}')

for run_name, r in results.items():
    print(f'{r["model"]:<15} {r["datasets"]:<15} {r["accuracy"]:<8.4f} {r["f1_score"]:<8.4f} {r["auc_roc"]:<8.4f} {r["sensitivity"]:<8.4f} {r["specificity"]:<8.4f}')

print(f'{"="*100}')